In [12]:
import numpy as np

In [13]:
# Version norma infinito

def autpotencias_inf(A, q_0, err=1e-10, m=500):
    q_tilde = A @ q_0
    rho_tilde = np.inf
    for k in range(m):
        j = np.argmax(np.abs(q_tilde))
        q = q_tilde/q_tilde[j]
        q_tilde = A @ q
        rho = q_tilde[j]

        if np.abs(rho - rho_tilde) < err:
            print("iteraciones:", k)
            return q, rho

        rho_tilde = rho

    return q, rho

# TEST
A_0 = np.random.random((3,3))
q_0 = np.random.random(3)

q_inf, rho_inf = autpotencias_inf(A_0, q_0)

print('----------------------------------------------------------------------------')
print(f"q_inf = {q_inf}")
print('----------------------------------------------------------------------------')
print(f"rho_inf = {rho_inf}")
print('----------------------------------------------------------------------------')
print(f"A@q_inf = {A_0@q_inf}")
print('----------------------------------------------------------------------------')
print(f"rho_inf*q_inf = {rho_inf*q_inf}")
print('----------------------------------------------------------------------------')
print(f"||A@q_inf - rho_inf*q_inf||_inf = {np.linalg.norm(A_0@q_inf - rho_inf*q_inf, np.inf)}")

iteraciones: 16
----------------------------------------------------------------------------
q_inf = [0.51262988 1.         0.71022938]
----------------------------------------------------------------------------
rho_inf = 1.6159458193837861
----------------------------------------------------------------------------
A@q_inf = [0.82838211 1.61594582 1.1476922 ]
----------------------------------------------------------------------------
rho_inf*q_inf = [0.82838211 1.61594582 1.1476922 ]
----------------------------------------------------------------------------
||A@q_inf - rho_inf*q_inf||_inf = 1.3776979557178493e-11


In [14]:
# Version norma 2

def autpotencias_2 (A, q_0, err=1e-10, M=500):
    q_tilde = A @ q_0
    potencia = np.linalg.norm(q_0, 2)**2
    rho_tilde = (q_0.T@q_tilde)/potencia
    for k in range(M):
        q = q_tilde/np.linalg.norm(q_tilde, 2)
        q_tilde = A @ q
        rho = q.T@q_tilde
        if np.abs(rho - rho_tilde) < err:
            print("iteraciones:", k)
            break
        rho_tilde = rho
    return q, rho

#TEST
A_0 = np.random.random((3,3))
q_0 = np.random.random(3)

q_2, rho_2 = autpotencias_2(A_0, q_0)

print('----------------------------------------------------------------------------')
print(f"q_2 = {q_2}")
print('----------------------------------------------------------------------------')
print(f"rho_2 = {rho_2}")
print('----------------------------------------------------------------------------')
print(f"A@q_2 = {A_0@q_2}")
print('----------------------------------------------------------------------------')
print(f"rho_2*q_2 = {rho_2*q_2}")
print('----------------------------------------------------------------------------')
print(f"||A@q_2 - rho_2*q_2||_2 = {np.linalg.norm(A_0@q_2 - rho_2*q_2, 2)}")

iteraciones: 13
----------------------------------------------------------------------------
q_2 = [0.71377014 0.50810721 0.48203657]
----------------------------------------------------------------------------
rho_2 = 1.8691436078866008
----------------------------------------------------------------------------
A@q_2 = [1.33413889 0.94972534 0.90099558]
----------------------------------------------------------------------------
rho_2*q_2 = [1.33413889 0.94972534 0.90099558]
----------------------------------------------------------------------------
||A@q_2 - rho_2*q_2||_2 = 1.7940644826986945e-10


In [15]:
#FUNCIONES AUXILIARES para el cociente de Rayleigh

def egaussp(A, b):
    m, n = A.shape
    U = A.copy()
    y = b.copy()

    for idx in range(min(m - 1, n)):
        if np.max(np.abs(U[idx:, idx])) != 0:
            # Elegimos el pivot
            pivot = idx + np.argmax(np.abs(U[idx:, idx]))
            # Pivoteamos (sin multiplicar por matriz elemental)
            U[[idx, pivot], :] = U[[pivot, idx], :]
            y[[idx, pivot]] = y[[pivot, idx]]
            # Reducir
            v = U[idx + 1:, idx] / U[idx, idx]
            U[idx + 1:, idx] = 0
            U[idx + 1:, idx + 1:] = U[idx + 1:, idx + 1:] - np.outer(v, U[idx, idx + 1:])
            y[idx + 1:] = y[idx + 1:] - v * y[idx]

    return U, y

def sol_trsupcol(A, b):
    n = len(b)
    x = b.copy()

    for idx in reversed(range(n)):
        if b[idx] != 0:
            j = idx
            break

    for i in reversed(range(j + 1)):
        x[i] = x[i] / A[i, i]
        x[:i] = x[:i] - A[:i, i] * x[i]

    return x

def sol_egauss(A, b):
    U, y = egaussp(A, b)
    x = sol_trsupcol(U, y)
    return x

In [16]:
# Version cociente de Rayleigh

def autrayleigh(A, q_0, err=1e-10, M=600):
    n = A.shape[0]
    Id = np.eye(n)
    q = q_0 /np.linalg.norm(q_0)
    q_tilde = q.copy()
    rho = q.T@A@q
    for k in range(M):
        z = sol_egauss(A - rho*Id, q_tilde)
        sigma = np.linalg.norm(z)
        q = z/sigma
        theta = np.dot(q, q_tilde) / sigma
        if np.abs(theta)< err:
            print("iteraciones", k)
            return q, rho+theta

        q_tilde = q
        rho = rho+theta

    return q, rho+theta

#TEST
A_0 = np.random.random((3,3))
q_0 = np.random.random(3)

q_r, rho_r = autrayleigh(A_0, q_0)

print('----------------------------------------------------------------------------')
print(f"Normalizado q_r = {q_r}")
print('----------------------------------------------------------------------------')
print(f"rho_r = {rho_r}")
print('----------------------------------------------------------------------------')
print(f"A@q_r = {A_0@q_r}")
print('----------------------------------------------------------------------------')
print(f"rho_r*q_r = {rho_r*q_r}")
print('----------------------------------------------------------------------------')
print(f"||A@q_r - rho_r*q_r||_2 = {np.linalg.norm(A_0@q_r - rho_r*q_r, 2)}")


iteraciones 5
----------------------------------------------------------------------------
Normalizado q_r = [0.65779199 0.4294995  0.61874056]
----------------------------------------------------------------------------
rho_r = 1.1413681354570742
----------------------------------------------------------------------------
A@q_r = [0.75078281 0.49021704 0.70621076]
----------------------------------------------------------------------------
rho_r*q_r = [0.75078281 0.49021704 0.70621076]
----------------------------------------------------------------------------
||A@q_r - rho_r*q_r||_2 = 1.2412670766236366e-16
